# 13 Batch Eye + OptiTrack Ingest Template

Template for routine multi-session eye and OptiTrack ingestion.


In [ ]:
import logging
import warnings

logging.getLogger("datajoint").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*", category=UserWarning)

from adamacs.notebook_runtime import bootstrap_ingest_notebook

ctx = bootstrap_ingest_notebook(verbose=False)
repo_root = ctx.repo_root

import datajoint as dj


In [ ]:

candidate_scans = (
    scan.Scan * session.Session * session.SessionUser * subject.User
    & f'initials = "{INITIALS}"'
    & f'session_datetime >= "{DATE_FROM}"'
).fetch("KEY")

print("candidate scans:", len(candidate_scans))
pd.DataFrame(candidate_scans).head(20)


In [ ]:

def has_rows(table, key):
    try:
        return len(table & key) > 0
    except Exception:
        return False

rows = []
for key in candidate_scans:
    rows.append(
        {
            **key,
            "has_behavior_recording": has_rows(event.BehaviorRecording, key),
            "has_video_recording": has_rows(model.VideoRecordingNew, key),
            "has_camsync": has_rows(behavior.CamSyncRecording, key),
            "has_optitrack": has_rows(mocap.MotionCapture, key),
            "has_rigidmouse": has_rows(virtual_markers_optitrack.RigidMouseTracking, key),
            "has_pupil_fit": has_rows(pupil_tracking.PupilEllipseFittingFreeMoving, key),
            "has_gaze3d": has_rows(pupil_tracking.GazeReconstruction3D, key),
        }
    )

coverage = pd.DataFrame(rows)
coverage


In [ ]:

missing_any = coverage[
    ~coverage[
        [
            "has_behavior_recording",
            "has_video_recording",
            "has_camsync",
            "has_optitrack",
            "has_rigidmouse",
            "has_pupil_fit",
            "has_gaze3d",
        ]
    ].all(axis=1)
]

missing_any


In [ ]:

# Optional population plan (disabled by default):
if ALLOW_DB_WRITES:
    raise RuntimeError("Set ALLOW_DB_WRITES manually after explicit approval.")

# Example execution order for write-enabled runs:
# 1) behavior.CamSyncRecording.populate(...)
# 2) mocap.MotionCapture.populate(...)
# 3) virtual_markers_optitrack.RigidMouseTracking.populate(...)
# 4) pupil_tracking.PupilEllipseFittingFreeMoving.populate(...)
# 5) pupil_tracking.GazeReconstruction3D.populate(...)
